In [ ]:
using LinearAlgebra

In [ ]:
#quaternion functions
function hat(v)
    return [0 -v[3] v[2];
            v[3] 0 -v[1];
            -v[2] v[1] 0]
end

function unhat(S)
    return 0.5*[S[3,2]-S[2,3];
                S[1,3]-S[3,1];
                S[2,1]-S[1,2]]
end

H = [zeros(1,3); I];
T = [1  zeros(1,3);
     zeros(3,1) -I];

function L(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I + hat(q[2:4])]
end

function R(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I - hat(q[2:4])]
end

function G(q)
    return L(q)*H
end

function Q(q)
    return H'*(R(q)'*L(q))*H
end

function expq(ϕ)
    θ = norm(ϕ)
    return [cos(θ); ϕ*sinc(θ/π)];
end

function logq(q)
    θ = acos(q[1])
    r = q[2:4]/norm(q[2:4])
    return θ*r
end

In [ ]:
#model parameters
J = Diagonal([1.0; 1.25; 1.5])
g = [1 -1 0  0;
     0  0 1 -1;
     0  0 0  0]
w = 0.1;

In [ ]:
function Bw(x)
    a = reshape(x[8:19],3,4)
    B = -w*[hat(g[:,1])*a[:,1] hat(g[:,2])*a[:,2] hat(g[:,3])*a[:,3] hat(g[:,4])*a[:,4]]
end

In [ ]:
#dynamics
function dynamics(x,u)
    q = x[1:4]/norm(x[1:4])
    ω = x[5:7]
    a = reshape(x[8:19],3,4)
    for k = 1:4
        a[:,k] .= a[:,k]/norm(a[:,k])
    end

    ρ = w*sum(a, dims=2)
    τ = Bw(x)*u
    
    q̇ = 0.5*G(q)*ω
    ω̇ = -J\(hat(ω)*(J*ω + ρ) - τ)
    ȧ = [hat(g[:,1])*a[:,1]*u[1];
         hat(g[:,2])*a[:,2]*u[2];
         hat(g[:,3])*a[:,3]*u[3];
         hat(g[:,4])*a[:,4]*u[4]]

    ẋ = [q̇; ω̇; ȧ]
end

In [ ]:
function controller(x)
    kp = 0.5;
    kd = 0.5;

    q = x[1:4]
    ϕ = logq(q)

    ω = x[5:7]
    
    τ = -kp*ϕ - kd*ω
end

In [ ]:
function steering(x,τ)
    B = Bw(x)
    #u = B'*((B*B')\τ)
    u = B'*((B*B'+1e-3*I)\τ)
end

In [ ]:
#Classic RK4 integrator: https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods
function rkstep(x,u)
    f1 = dynamics(x,u)
    f2 = dynamics(x + 0.5*h*f1,u)
    f3 = dynamics(x + 0.5*h*f2,u)
    f4 = dynamics(x + h*f3,u)
    xn = x + (h/6.0)*(f1 + 2*f2 + 2*f3 + f4)
    
    xn[1:4] .= xn[1:4]/norm(xn[1:4]) #re-normalize quaternion

    #re-normalize wheel axes
    xn[8:10] .= xn[8:10]/norm(xn[8:10])
    xn[11:13] .= xn[11:13]/norm(xn[11:13])
    xn[14:16] .= xn[14:16]/norm(xn[14:16])
    xn[17:19] .= xn[17:19]/norm(xn[17:19])
    
    return xn
end

In [ ]:
h = 0.1 #time step
n = 1000 #number of time steps
tf = n*h #final time

#sample random attitude
q0 = expq(0.2*randn(3))

ω0 = 0.01*randn(3)

#random initial gimbal angles
a0 = zeros(3,4)
for k = 1:4
    a0[:,k] .= randn(3)
    a0[:,k] .= hat(g[:,k])*a0[:,k]
    a0[:,k] .= a0[:,k]/norm(a0[:,k])
end

x0 = [q0; ω0; a0[:]];

In [ ]:
#example initial conditions that hit singularity
x0 = [0.847226561031405;
  0.0732446800051401;
  0.5258364795266754;
  0.018394780043152908;
 -0.002807278981679324;
  0.020918484382436424;
 -0.00798171207723605;
  0.0;
  0.13174195145061054;
  0.9912840451797784;
  0.0;
  0.9661136333509235;
  0.2581171196443148;
  0.1061036485387192;
  0.0;
 -0.9943550752959286;
  0.8187169663533636;
  0.0;
 -0.5741972910116743];

In [ ]:
#Simulate n time steps
xhist = zeros(19,n)
xhist[:,1] .= x0

Bcond = zeros(n)
Bcond[1] = cond(Bw(x0))

#u = zeros(4) #no gimble rates
for k = 1:(n-1)
    u = steering(xhist[:,k], controller(xhist[:,k]))
    xhist[:,k+1] = rkstep(xhist[:,k],u)
    Bcond[k+1] = cond(Bw(xhist[:,k+1]))
end

In [ ]:
using MeshCat, GeometryBasics, CoordinateTransformations, Rotations

In [ ]:
vis = Visualizer()
setobject!(vis[:box1], Rect(Vec(-1.5, -1, -0.5), Vec(3, 2, 1)))

In [ ]:
render(vis)

In [ ]:
anim = Animation(vis)

for t = 1:n
    atframe(anim, t) do
        settransform!(vis[:box1], LinearMap(QuatRotation(xhist[1,t], xhist[2,t], xhist[3,t], xhist[4,t])))
    end
end

setanimation!(vis, anim)

In [ ]:
using PythonPlot

In [ ]:
plot(xhist[1,:])
plot(xhist[2,:])
plot(xhist[3,:])
plot(xhist[4,:])

In [ ]:
plot(xhist[5,:])
plot(xhist[6,:])
plot(xhist[7,:])

In [ ]:
plot(xhist[8,:])
plot(xhist[9,:])
plot(xhist[10,:])

In [ ]:
plot(xhist[11,:])
plot(xhist[12,:])
plot(xhist[13,:])

In [ ]:
plot(xhist[14,:])
plot(xhist[15,:])
plot(xhist[16,:])

In [ ]:
plot(xhist[17,:])
plot(xhist[18,:])
plot(xhist[19,:])

In [ ]:
plot(Bcond)